In [1]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import pandas as pd
import numpy as np

# Load cleaned dataset
file_path = "Cleaned_Merged_data_alarm.csv"
df = pd.read_csv(file_path)

# Pisahkan fitur dan target
X = df.drop(columns=["Severity", "Alarm Serial Number"])  # Fitur
y = df["Severity"]  # Target

# Normalisasi fitur numerik
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Splitting Data (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

# Mengatasi Ketidakseimbangan Kelas dengan SMOTE
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Tampilkan distribusi kelas setelah SMOTE
severity_distribution_resampled = np.bincount(y_train_resampled)
severity_distribution_original = np.bincount(y_train)

print("Distribusi Severity sebelum SMOTE:", severity_distribution_original)
print("Distribusi Severity setelah SMOTE:", severity_distribution_resampled)


Distribusi Severity sebelum SMOTE: [38232  4580  9206 26227]
Distribusi Severity setelah SMOTE: [38232 38232 38232 38232]


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# Load cleaned dataset
file_path = "Cleaned_Merged_data_alarm.csv"
df = pd.read_csv(file_path)

# Pisahkan fitur dan target
X = df.drop(columns=["Severity", "Alarm Serial Number"])
y = df["Severity"]

# Normalisasi fitur numerik
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Splitting Data (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

# Mengatasi Ketidakseimbangan Kelas dengan SMOTE
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Simpan dataset yang telah diproses
np.save("X_train.npy", X_train_resampled)
np.save("y_train.npy", y_train_resampled)
np.save("X_test.npy", X_test)
np.save("y_test.npy", y_test)

print("Dataset train-test telah berhasil disimpan ulang.")


Dataset train-test telah berhasil disimpan ulang.


In [3]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# Load processed train-test datasets
X_train = np.load("X_train.npy")
y_train = np.load("y_train.npy")
X_test = np.load("X_test.npy")
y_test = np.load("y_test.npy")

# Reshape for LSTM (samples, timesteps, features)
X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

# Build LSTM model
model = Sequential([
    LSTM(64, activation='relu', return_sequences=True, input_shape=(1, X_train.shape[2])),
    Dropout(0.2),
    BatchNormalization(),
    LSTM(32, activation='relu', return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(4, activation='softmax')  # 4 classes for severity levels
])

# Compile model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train model with Early Stopping
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=50, batch_size=32, callbacks=[early_stopping])

# Evaluate model
y_pred = np.argmax(model.predict(X_test), axis=1)
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

# Print evaluation results
print(f"Model Accuracy: {accuracy * 100:.2f}%")
print("Classification Report:\n", report)

# Save model
model.save("lstm_alarm_model.h5")
print("LSTM model saved as lstm_alarm_model.h5")


2025-05-23 14:34:36.976842: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-23 14:34:36.979803: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-23 14:34:36.987575: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747985677.000177  129122 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747985677.004039  129122 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-23 14:34:37.017389: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU ins

Epoch 1/50


2025-05-23 14:34:38.955490: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/home/ubuntu/ryh_training/venv/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


4779/4779 ━━━━━━━━━━━━━━━━━━━━ 13s 2ms/step - accuracy: 0.7964 - loss: 0.5198 - val_accuracy: 0.8967 - val_loss: 0.3088
Epoch 2/50
4779/4779 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - accuracy: 0.8839 - loss: 0.3094 - val_accuracy: 0.8984 - val_loss: 0.2961
Epoch 3/50
4779/4779 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - accuracy: 0.8925 - loss: 0.2889 - val_accuracy: 0.9070 - val_loss: 0.2539
Epoch 4/50
4779/4779 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - accuracy: 0.8966 - loss: 0.2734 - val_accuracy: 0.9009 - val_loss: 0.2654
Epoch 5/50
4779/4779 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - accuracy: 0.9023 - loss: 0.2620 - val_accuracy: 0.9041 - val_loss: 0.2486
Epoch 6/50
4779/4779 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - accuracy: 0.9024 - loss: 0.2567 - val_accuracy: 0.9041 - val_loss: 0.2464
Epoch 7/50
4779/4779 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - accuracy: 0.9067 - loss: 0.2476 - val_accuracy: 0.9054 - val_loss: 0.2458
Epoch 8/50
4779/4779 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - accuracy: 0.9098 - loss: 0.2432 - val

Model Accuracy: 92.47%
Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.88      0.93      9558
           1       0.83      0.88      0.85      1145
           2       0.71      0.97      0.82      2302
           3       0.98      0.98      0.98      6557

    accuracy                           0.92     19562
   macro avg       0.87      0.93      0.90     19562
weighted avg       0.94      0.92      0.93     19562

LSTM model saved as lstm_alarm_model.h5
